# SynBioCrow 2.4.17 — Development Literature-Anchor Evaluation
Upload only the 2.4.17 SOURCE ZIP. This evaluates the frozen 2.4.16 ranking against literature-supported pathway anchors for the four development targets. Validation records remain sealed.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,os,shutil,subprocess,sys,time,zipfile
drive.mount("/content/drive",force_remount=False)
T0=time.time();BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4")
REPO=Path("/content/SynBioCrow_2_4_17");SRC=Path("/content/src2417")
RUN=Path("/content/synbiocrow_2_4_17");OUT=BASE/"development_truth_2_4_17";OUT.mkdir(parents=True,exist_ok=True)
def st(p,m):print(f"[2.4.17] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)
def run(cmd,cwd=None,timeout=1800):
 print("$"," ".join(map(str,cmd)),flush=True);e=os.environ.copy()
 if cwd:e["PYTHONPATH"]=str(cwd)+os.pathsep+e.get("PYTHONPATH","")
 q=subprocess.run(list(map(str,cmd)),cwd=cwd,env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
 print(q.stdout,flush=True)
 if q.returncode:raise subprocess.CalledProcessError(q.returncode,cmd,output=q.stdout)
 return q
st(3,"upload 2.4.17 SOURCE ZIP only");u=files.upload()
if len(u)!=1:raise RuntimeError("Upload exactly one SOURCE ZIP")
name=next(iter(u));z=Path("/content")/name;z.write_bytes(u[name]);print("SOURCE SHA256",hashlib.sha256(z.read_bytes()).hexdigest())
st(8,"clone develop/2.4");shutil.rmtree(REPO,ignore_errors=True)
run(["git","clone","--depth","1","--branch","develop/2.4","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)
st(14,"overlay source");shutil.rmtree(SRC,ignore_errors=True);SRC.mkdir(parents=True)
with zipfile.ZipFile(z) as q:q.extractall(SRC)
for p in SRC.rglob("*"):
 if p.is_file() and p.name not in {"README_SOURCE.txt","SOURCE_MANIFEST.json"}:
  d=REPO/p.relative_to(SRC);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,d)
st(20,"install core + RDKit");run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)
man=json.loads((REPO/"benchmarks/2.4/split_manifest_2_4_11.json").read_text())
dev=[r["record_id"] for r in man["records"] if r["split"]=="development"];sealed=[r["record_id"] for r in man["records"] if r["split"]!="development"]
st(28,"discover complete development candidates and frozen 2.4.16 ranking")
dirs=sorted([p for p in BASE.glob("development_routes_*") if p.is_dir()],key=lambda p:p.stat().st_mtime,reverse=True)
inp=next((d for d in dirs[:20] if all((d/f"{rid}.json").is_file() for rid in dev)),None)
if inp is None:raise FileNotFoundError("complete development route directory not found")
ranked=BASE/"native_discrimination_2_4_16"/"native_enriched_ranked_routes.json"
if not ranked.is_file():raise FileNotFoundError(ranked)
print("INPUT",inp);print("RANKING",ranked);print("SEALED / NOT OPENED",sealed)
st(40,"evaluate development literature anchors");shutil.rmtree(RUN,ignore_errors=True);RUN.mkdir(parents=True)
cache=OUT/"pubchem_anchor_cache.json"
cmd=[sys.executable,str(REPO/"scripts/v24_evaluate_development_truth_2_4_17.py"),
 "--input-dir",str(inp),"--ranked-json",str(ranked),
 "--truth",str(REPO/"benchmarks/2.4/development_truth_anchors_2_4_17.json"),
 "--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
 "--out-dir",str(RUN)]
if cache.is_file():cmd+=["--cache",str(cache)]
run(cmd,cwd=REPO,timeout=1800)
st(72,"verify firewall")
s=json.loads((RUN/"development_anchor_summary.json").read_text())
assert s["validation_truth_accessed"] is False and s["evaluation_truth_accessed"] is False
assert s["ranking_weights_fit"] is False and set(s["development_record_ids"])==set(dev)
print(json.dumps(s,indent=2),flush=True)
st(86,"persist + bundle")
for p in RUN.iterdir():
 if p.is_file():shutil.copy2(p,OUT/p.name)
bundle=OUT/"SynBioCrow_2_4_17_DEVELOPMENT_TRUTH_EVALUATION_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as q:
 for p in sorted(OUT.iterdir()):
  if p.is_file() and p!=bundle:q.write(p,p.name)
st(100,"PASS");print("BUNDLE",bundle);print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest());files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/development_truth_2_4_17/SynBioCrow_2_4_17_DEVELOPMENT_TRUTH_EVALUATION_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
